## Import package

In [ ]:
import sys
sys.path.append('..')

import json
import os
import numpy as np
import matplotlib.pyplot as plt
from src import create_data_generators, get_class_weights
from src import build_transfer_model
from src import train_model
from src import evaluate_model, plot_confusion_matrix, plot_roc_curve, plot_training_history

os.makedirs('../models', exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'legend.facecolor': 'white',
    'legend.edgecolor': 'black'
})

## Load data with augmentation (150x150)

In [ ]:
# 150x150 outperformed 224x224 in experiments — less noise, better generalization
data_dir = '../data/chest_xray'
target_size = (150, 150)

train_gen, val_gen, test_gen = create_data_generators(
    data_dir, target_size=target_size, batch_size=32, augment=True
)

## Get class weights

In [ ]:
class_weights = get_class_weights(train_gen)

## Build VGG16 transfer learning model (frozen base)

In [ ]:
model, base_model = build_transfer_model(
    input_shape=(150, 150, 3),
    base_model_name="vgg16",
    learning_rate=0.0001,
    fine_tune_layers=0  # All base layers frozen
)

model.summary()

## Phase 1 — Train classification head only

In [ ]:
# VGG16's pre-trained layers stay frozen
# Only our custom Dense layers are trained
# This teaches the model: given VGG16's features, is this Normal or Pneumonia?
print("PHASE 1: Training classification head only")
print("=" * 40)

history_phase1 = train_model(
    model,
    train_gen, val_gen,
    epochs=20,
    class_weights=class_weights,
    model_save_path='../models/vgg16_150_phase1.keras'
)

## Plot Phase 1 training history

In [ ]:
plot_training_history(history_phase1, title="VGG16 150x150 - Phase 1 (Frozen)")

## Evaluate Phase 1

In [ ]:
results_phase1 = evaluate_model(model, test_gen)
print(f"\nPhase 1 accuracy: {results_phase1['accuracy'] * 100:.2f}%")

## Phase 2 — Fine-tune top layers

In [ ]:
# Unfreeze last 8 layers of VGG16 for fine-tuning
# VGG16 has fewer layers than ResNet50, so 8 is a good number
# This lets the model slightly adjust pre-trained features
# to better match chest X-ray patterns
print("\nPHASE 2: Fine-tuning top layers")
print("=" * 40)

base_model.trainable = True
for layer in base_model.layers[:-8]:
    layer.trainable = False

trainable = sum(1 for layer in model.layers if layer.trainable)
frozen = sum(1 for layer in model.layers if not layer.trainable)
print(f"Trainable layers: {trainable}, Frozen layers: {frozen}")

# Recompile with a lower learning rate for fine-tuning
# Lower LR prevents destroying the pre-trained weights
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## Train Phase 2

In [ ]:
history_phase2 = train_model(
    model,
    train_gen, val_gen,
    epochs=20,
    class_weights=class_weights,
    model_save_path='../models/vgg16_150.keras'
)

## Plot Phase 2 training history

In [ ]:
plot_training_history(history_phase2, title="VGG16 150x150 - Phase 2 (Fine-Tuned)")

## Evaluate fine-tuned model on test set

In [ ]:

results = evaluate_model(model, test_gen)

## Confusion matrix

In [ ]:
plot_confusion_matrix(
    results['y_true'], results['y_pred'],
    results['class_names'],
    title="VGG16 150x150 - Confusion Matrix"
)

## ROC curve

In [ ]:
auc_score = plot_roc_curve(
    results['y_true'], results['y_prob'],
    title="VGG16 150x150 - ROC Curve"
)

## Compare with previous models

In [ ]:
with open('../models/baseline_results.json', 'r') as f:
    baseline = json.load(f)
with open('../models/transfer_results.json', 'r') as f:
    resnet = json.load(f)

print("\nMODEL COMPARISON:")
print(f"{'Metric':<12} {'Baseline':<12} {'ResNet50':<12} {'VGG16':<12}")
print("-" * 48)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    b_val = baseline[metric]
    r_val = resnet[metric]
    v_val = results[metric] if metric != 'auc' else auc_score
    best = "***" if v_val >= max(b_val, r_val) else ""
    print(f"{metric:<12} {b_val:<12.4f} {r_val:<12.4f} {v_val:<12.4f} {best}")

## Save VGG16 results and auto-select best model

In [ ]:
import shutil

vgg16_results = {
    "model": "vgg16_150x150",
    "accuracy": results['accuracy'],
    "precision": results['precision'],
    "recall": results['recall'],
    "f1": results['f1'],
    "auc": auc_score
}

with open('../models/vgg16_results.json', 'w') as f:
    json.dump(vgg16_results, f, indent=2)

# Auto-select best model across all three
all_models = {
    "baseline_cnn": (baseline, '../models/baseline_model.keras'),
    "resnet50": (resnet, '../models/transfer_model.keras'),
    "vgg16_150x150": (vgg16_results, '../models/vgg16_150.keras')
}

best_name = max(all_models, key=lambda k: all_models[k][0]['accuracy'])
best_results, best_file = all_models[best_name]

if os.path.exists(best_file):
    shutil.copy(best_file, '../models/best_model.keras')
    best_config = {
        "source": best_name,
        **best_results
    }
    with open('../models/best_config.json', 'w') as f:
        json.dump(best_config, f, indent=2)
    print(f"\nBest model: {best_name} ({best_results['accuracy'] * 100:.2f}%)")
    print("Saved to: ../models/best_model.keras")
else:
    print(f"Warning: {best_file} not found")

## Summary

In [ ]:
print("=" * 50)
print("VGG16 TRANSFER LEARNING SUMMARY")
print("=" * 50)
print(f"Base model: VGG16")
print(f"Image size: {target_size[0]}x{target_size[1]}")
print(f"Phase 1 (frozen): {results_phase1['accuracy'] * 100:.2f}% accuracy")
print(f"Phase 2 (fine-tuned): {results['accuracy'] * 100:.2f}% accuracy")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall:    {results['recall']:.4f}")
print(f"F1-Score:  {results['f1']:.4f}")
print(f"AUC:       {auc_score:.4f}")
print("=" * 50)
print(f"\nImprovement over baseline: {(results['accuracy'] - baseline['accuracy']) * 100:+.2f}%")
print(f"Improvement over ResNet50: {(results['accuracy'] - resnet['accuracy']) * 100:+.2f}%")
print("\nNext: 05_evaluation.ipynb")